# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook builds a leakage-aware contract for the Refresh / Content Opportunity Scoring lane. It uses March 2026 as the feature month and a strictly later April–May outcome window. March is a mid-panel iteration month; the final June 2026 month remains sealed and is not used to develop label logic.

**Execution requirement:** dataset access must be approved and a READ token must be available as the `HF_TOKEN` environment variable or Colab Secret. Never paste the token into this notebook. Outputs are produced only by running the queries against the hosted warehouse.

## 1. The contract, in plain words (five answers)

1. **One row:** one `(client_hash_id, content_hash_id)` page for the March feature window; the source fact grain is one page-client-date.
2. **Table:** `fact_content_daily_performance` only. No query table or product-generated flags are needed for this first pass.
3. **Time window:** features use 30 days, 2026-03-02 through 2026-03-31. Assume a seven-day reporting buffer and decide on 2026-04-07. The 30-day outcome window is 2026-04-08 through 2026-05-07, after the decision point.
4. **Target/proxy:** flag a page when April 8–May 7 GSC impressions are more than 20% below the March 2–31 baseline, requiring at least 100 March impressions and GSC availability in both windows. This is an observed future-window proxy, not a causal refresh effect.
5. **Deliberately excluded:** client/content IDs are grouping keys only; future-window impressions, their ratio, and all label-derived fields are excluded from the honest five-feature model because they are unavailable at the 2026-04-07 decision point.

**Decision supported:** give an SEO/content editor a ranked review queue for pages to inspect for a possible refresh. A score is not an automatic instruction to edit.

In [ ]:
%pip -q install duckdb scikit-learn


In [ ]:
import os
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN', default=None)
    except ImportError:
        HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError(
        'HF_TOKEN is unavailable. Request access to FlyRank/internship-warehouse, '
        'then set a READ token as an environment variable or Colab Secret named HF_TOKEN.'
    )

con = duckdb.connect()
escaped_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
del escaped_token
del HF_TOKEN

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
APRIL_PATH = f"{REL}/fact_content_daily_performance/month=2026-04/*.parquet"
MAY_PATH = f"{REL}/fact_content_daily_performance/month=2026-05/*.parquet"
print('Connected to the gated warehouse. Features: March 2026; outcome window: April 8–May 7, 2026.')

## 2. Fields: feature / label / context / excluded

- **Feature (five maximum):** March impressions, March clicks, March days with impressions, March days with clicks, and March mean position. These use only report dates in March and are intended to be available for a 2026-04-07 decision after a seven-day reporting buffer. Zero positions are excluded from the mean because zero is not a real rank.
- **Label:** impressions in 2026-04-08 through 2026-05-07 are more than 20% below the March 2–31 baseline, using GSC-available rows and requiring at least 100 March impressions.
- **Context:** `client_hash_id` groups train/test splits; `(client_hash_id, content_hash_id)` identifies an item. Neither ID is a model feature.
- **Excluded:** future-window impressions and their ratio are outcome-period information. The ratio is deliberately introduced once as the leakage trap, scored, and then removed. `ga4_data_available` is checked for coverage but GA4 values are not used in this GSC-only five-feature frame.

**Availability note:** availability flags are three-valued. In particular, use `ga4_data_available IS TRUE`; `FALSE` and `NULL` are not equivalent to measured zero.

## 3. Verify with exactly three warehouse queries

These are the three contract verification queries. Query 1 checks duplicate page-client-date keys; query 2 records the March row count and date span; query 3 measures rows with GA4 explicitly available. Run them on the mid-panel month partition, not the final-month sample.

In [ ]:
# Verification query 1/3 — grain probe. An empty result means no duplicate fact keys were found.
grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_key
    FROM read_parquet('{MARCH_PATH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f'Duplicate page-client-date keys returned: {len(grain_probe)} (expected 0)')
display(grain_probe)
assert grain_probe.empty, 'The declared daily fact grain has duplicate keys in this slice.'

In [ ]:
# Verification query 2/3 — row count and actual date span for the March partition.
march_summary = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_report_date,
           MAX(report_date) AS max_report_date
    FROM read_parquet('{MARCH_PATH}')
""").df()
display(march_summary)
assert int(march_summary.loc[0, 'row_count']) > 0, 'March partition returned no rows.'

In [ ]:
# Verification query 3/3 — count rows where GA4 availability is explicitly TRUE.
availability = con.sql(f"""
    SELECT COUNT(*) AS march_fact_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS not_explicitly_available_rows
    FROM read_parquet('{MARCH_PATH}')
""").df()
display(availability)
assert int(availability.loc[0, 'ga4_available_rows']) > 0, 'No rows have GA4 availability explicitly TRUE.'

## 4. Build the five-feature frame and perform the leakage trap

Each feature uses GSC measurements from March 2–31. The assumed decision point is 2026-04-07, after a seven-day reporting buffer. The label outcome window is 2026-04-08 through 2026-05-07, strictly after the decision point. We require at least 100 impressions in the March baseline to avoid treating a tiny denominator as a stable decline signal.

In [ ]:
feature_frame = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS march_impressions,
               SUM(gsc_clicks) AS march_clicks,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS march_days_with_impressions,
               COUNT(DISTINCT CASE WHEN gsc_clicks > 0 THEN report_date END) AS march_days_with_clicks,
               AVG(NULLIF(gsc_avg_position, 0)) AS march_mean_position
        FROM read_parquet('{MARCH_PATH}')
        WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100
    ),
    outcome_rows AS (
        SELECT client_hash_id, content_hash_id, report_date, gsc_impressions
        FROM read_parquet('{APRIL_PATH}')
        WHERE report_date BETWEEN DATE '2026-04-08' AND DATE '2026-04-30'
          AND gsc_data_available IS TRUE
        UNION ALL
        SELECT client_hash_id, content_hash_id, report_date, gsc_impressions
        FROM read_parquet('{MAY_PATH}')
        WHERE report_date BETWEEN DATE '2026-05-01' AND DATE '2026-05-07'
          AND gsc_data_available IS TRUE
    ),
    outcome AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS outcome_impressions
        FROM outcome_rows
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) IS NOT NULL
    )
    SELECT march.client_hash_id,
           march.content_hash_id,
           march.march_impressions,
           march.march_clicks,
           march.march_days_with_impressions,
           march.march_days_with_clicks,
           march.march_mean_position,
           outcome.outcome_impressions,
           outcome.outcome_impressions / march.march_impressions AS future_to_march_impression_ratio_leak,
           CAST(outcome.outcome_impressions < 0.8 * march.march_impressions AS INTEGER) AS future_decline_label
    FROM march
    INNER JOIN outcome USING (client_hash_id, content_hash_id)
""").df()

feature_cols = [
    'march_impressions',
    'march_clicks',
    'march_days_with_impressions',
    'march_days_with_clicks',
    'march_mean_position',
]
label_col = 'future_decline_label'
assert len(feature_cols) == 5
assert not set(feature_cols) & {'outcome_impressions', 'future_to_march_impression_ratio_leak'}
assert not feature_frame.duplicated(['client_hash_id', 'content_hash_id']).any()
assert len(feature_frame) > 0, 'No page-client rows have eligible features and both observed future windows.'
print(f'Page-client rows with eligible March features and observed future windows: {len(feature_frame):,}')
print(f"Future decline-proxy rate in this eligible frame: {feature_frame[label_col].mean():.1%}")
display(feature_frame[feature_cols + [label_col]].head())

### Feature timing check — one line each

- `march_impressions`: knowable by the 2026-04-07 decision point, assuming the seven-day reporting buffer is sufficient, because it sums GSC impressions from March 2–31.
- `march_clicks`: knowable by the 2026-04-07 decision point under the same reporting-lag assumption, because it sums GSC clicks from March 2–31.
- `march_days_with_impressions`: knowable by the decision point under that assumption because it counts March dates with positive GSC impressions.
- `march_days_with_clicks`: knowable by the decision point under that assumption because it counts March dates with positive GSC clicks.
- `march_mean_position`: knowable by the decision point under that assumption because it averages March GSC positions, treating zero as missing.

`future_to_march_impression_ratio_leak` is not a valid feature: it uses the future outcome-period impressions used to create the label.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

y = feature_frame[label_col].astype('int8')
groups = feature_frame['client_hash_id']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(feature_frame[feature_cols], y, groups=groups))
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
assert y_train.nunique() == 2 and y_test.nunique() == 2, 'Held-out client split must contain both label classes.'

# Deliberate trap: this ratio includes the future outcome period and is the exact quantity thresholded to create y.
leak_model = make_pipeline(
    SimpleImputer(strategy='median'),
    DecisionTreeClassifier(max_depth=1, random_state=42),
)
leak_X = feature_frame[['future_to_march_impression_ratio_leak']]
leak_model.fit(leak_X.iloc[train_idx], y_train)
leak_probability = leak_model.predict_proba(leak_X.iloc[test_idx])[:, 1]
leak_accuracy = accuracy_score(y_test, leak_model.predict(leak_X.iloc[test_idx]))
leak_auc = roc_auc_score(y_test, leak_probability)
print(f'LEAKY score (do not report as model performance): accuracy={leak_accuracy:.3f}; ROC-AUC={leak_auc:.3f}')

# Remove the trap column before the honest model is trained.
feature_frame.drop(columns=['future_to_march_impression_ratio_leak', 'outcome_impressions'], inplace=True)
assert 'future_to_march_impression_ratio_leak' not in feature_frame.columns
assert 'outcome_impressions' not in feature_frame.columns

honest_model = make_pipeline(
    SimpleImputer(strategy='median'),
    RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),
)
honest_model.fit(feature_frame[feature_cols].iloc[train_idx], y_train)
honest_probability = honest_model.predict_proba(feature_frame[feature_cols].iloc[test_idx])[:, 1]
honest_prediction = honest_model.predict(feature_frame[feature_cols].iloc[test_idx])
honest_accuracy = accuracy_score(y_test, honest_prediction)
honest_auc = roc_auc_score(y_test, honest_probability)
print(f'HONEST five-feature score (held-out clients): accuracy={honest_accuracy:.3f}; ROC-AUC={honest_auc:.3f}')
print(f'Held-out positive-label prevalence: {y_test.mean():.3f}')
print('The leak feature and future outcome count have been deleted from feature_frame.')

## 5. One named limitation

**Coverage/selection limitation:** the eligible frame includes only page-client pairs with at least 100 GSC impressions in the March feature window and explicitly available GSC data in March and the April–May outcome window. It excludes low-volume pages and pairs with no observed outcome-period row, so results do not describe the full inventory. The warehouse has report dates but no ingestion timestamps in this contract, so the seven-day reporting buffer is an assumption, not a verified historical availability guarantee. It is also one time split; seasonality or client-specific history can change the pattern. This is an instructional proxy, not a claim that refreshes cause recovery.

The split holds out clients, but this single month and single split do not establish stable performance across time. A later model needs more time periods, a time-aware evaluation, and comparison with a transparent rule baseline.

## 6. Self-check

- [x] Five plain-language contract answers are stated
- [x] Exactly three verification query cells check grain, row count/date span, and availability with `IS TRUE`
- [x] Five March-only features have a knowability explanation
- [x] A future-period ratio is deliberately scored, then removed before the held-out-client honest score
- [x] One specific coverage limitation is named
- [ ] Run top to bottom with approved HF access; inspect all three real query outputs and both scores
- [ ] Commit this notebook after the hosted execution has completed successfully